# PSTAT100 Final Project Report
**Author:** Matthew Kim
**Date:** 2026-05-06


___

#### Topic Outline:

- Final project report (Interactive Jupyter notebook):
  - Abstract, introduction, methodology, data preparation & EDA, results, conclusion, references.
- **Data:** UCI Adult / Census Income (`database/adult.data`, `adult.test`).
- **Outcome:** predict income **> \$50K** vs **<= \$50K** (binary classification).
- **Models:** logistic regression (`sklearn`) vs random forest comparison on the official train/test split.

---

#### Collaboration

You are encouraged to collaborate with other students in your labs, but you are expected to write up your own work for submission. Please do not copy and paste other people's solutions to problems as it is considered plagiarism and you will be penalized and reported. Should you choose to collaborate with others, please note their names here:

**Your name:** Matthew Kim

**Collaborators:**

1.

___

#### Agent Usage

Additionally, you are permitted to use resources such as ChatGPT and Claude to help you with your lab assignments and to enhance your learning experience. Please make a note of any agents you have used in this submission here:

**Agents:** Cursor / Claude (notebook scaffolding and modeling suggestions)

---


## 1. Abstract *(target ~250 words)*

This notebook investigates the **UCI Adult** data set, a sample of census-derived records used to benchmark classification algorithms. The motivating question is whether demographic and employment variables help predict whether an individual is labeled as earning **more than \$50,000 per year** versus **at most \$50,000**. We document where the data come from, treat `?` entries as missing, and explore relationships between predictors and the outcome using summary tables and graphics consistent with **exploratory data analysis** practices from lecture and labs. For modeling we fit **logistic regression** within a reproducible `sklearn` pipeline with imputation, scaling, and one-hot encoding, and we compare performance to a **random forest** ensemble. We evaluate predictions on the repository's held-out `adult.test` split using accuracy, ROC-AUC, and confusion matrices. **Expand this section with your final motivation, methods, and numerical results once you finish polishing the narrative.**

---


## 2. Introduction

This report follows the structure required for the course final project: background and roadmap (**Section 2**), mathematical setup (**Section 3**), data sourcing and interactive preparation (**Section 4**), estimation results (**Section 5**), synthesis (**Section 6**), and citations (**Section 7**).

The Adult prediction task is a standard illustration of **supervised classification**. Findings describe associations in a historical extract and should not be read as causal effects of any single variable.

---


## 3. Methodology

### 3.1 Problem formulation

Let \(y_i \in \{0,1\}\) encode whether record \(i\) has label `<=50K` vs `>50K`. We observe a feature vector \(\mathbf{x}_i\) mixing quantitative and categorical inputs. We estimate the conditional probability \(\mathbb{P}(y=1 \mid \mathbf{x})\).

### 3.2 Logistic regression

**Logistic regression** specifies \(\mathbb{P}(y=1 \mid \mathbf{x}) = \sigma(\eta(\mathbf{x}))\) where \(\sigma\) is the logistic function and \(\eta\) is linear in the transformed (encoded) features. In `sklearn`, `LogisticRegression` applies **L2 regularization** by default, stabilizing estimates when many dummy variables are created by one-hot encoding.

### 3.3 Random forest *(comparison model)*

A **random forest** averages predictions from many decision trees fit on bootstrap samples; it can capture nonlinearities and interactions. We include it as a simple, strong baseline to contrast with the linear logit.

### 3.4 Evaluation

On the fixed **test** partition we report **accuracy**, **ROC-AUC**, `classification_report`, the **confusion matrix**, and an **ROC curve**. This mirrors the idea from lecture/labs that graphics and concise summaries beat dumping raw console output.

---


## 4. Data

### 4.1 Source

The **Adult** data were donated to the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/2/adult) (Becker & Kohavi). Files in `database/` mirror that distribution: `adult.data` (training), `adult.test` (test), and `adult.names` (variable descriptions).

### 4.2 Working directory

Open this notebook from the `final project` folder (same level as `database/`) so paths like `database/adult.data` resolve, matching how labs load `data/student_performance.csv` relative to the notebook location.

### 4.3 Packages

The next chunk imports libraries used across **Lab 4**, **Lab 5**, and **Assignment 1** (`numpy`, `pandas`, `matplotlib`, `seaborn`, `missingno`) plus `sklearn` for modeling.


In [ ]:
# Packages
import numpy as np
import pandas as pd
import missingno as msno
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    classification_report,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Seaborn style (same as Lab 4 / Lab 5)
sns.set_style("whitegrid")
sns.set_palette("Set2")

rng = np.random.RandomState(42)


In [ ]:
from pathlib import Path

DATA_DIR = Path("database")
assert DATA_DIR.exists(), "Run from the `final project` folder so ./database exists"

COLS = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education-num",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital-gain",
    "capital-loss",
    "hours-per-week",
    "native-country",
    "income",
]

NUMERIC = [
    "age",
    "fnlwgt",
    "education-num",
    "capital-gain",
    "capital-loss",
    "hours-per-week",
]
CATEGORICAL = [
    "workclass",
    "education",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native-country",
]


def load_adult(data_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    train = pd.read_csv(
        data_dir / "adult.data",
        names=COLS,
        skipinitialspace=True,
        na_values="?",
    )
    test = pd.read_csv(
        data_dir / "adult.test",
        names=COLS,
        skipinitialspace=True,
        na_values="?",
        skiprows=1,
    )
    test["income"] = test["income"].str.replace(".", "", regex=False).str.strip()
    return train, test


train_df, test_df = load_adult(DATA_DIR)


### 4.4 First look (`shape`, `info`, `head`)

As in the labs, we inspect dimensions, dtypes, and the first rows before modeling.


In [ ]:
print(train_df.shape, test_df.shape)
print(train_df.info())
train_df.head()


### 4.5 Target encoding and feature matrices

We encode the label as 1 if `income == '>50K'` and 0 otherwise, then drop `income` from predictors.


In [ ]:
LABEL_POS = ">50K"


def encode_target(s: pd.Series) -> pd.Series:
    return (s.str.strip() == LABEL_POS).astype(int)


y_train = encode_target(train_df["income"])
y_test = encode_target(test_df["income"])

X_train = train_df.drop(columns=["income"])
X_test = test_df.drop(columns=["income"])

print("Positive rate — train:", round(y_train.mean(), 4))
print("Positive rate — test:", round(y_test.mean(), 4))


### 4.6 Missing values

UCI codes unknown categories as `?`; `read_csv` maps those to `NaN`. Below we visualize **patterns of missingness** on the training predictors using `missingno`, which Lab 4 / Lab 5 import for this purpose.

For modeling we **impute** inside `sklearn` pipelines (median for numeric features, most frequent for categorical) so test-set values are never used when fitting transforms on the training data.


In [ ]:
missing_frac = X_train.isna().mean().sort_values(ascending=False)
missing_frac[missing_frac > 0].to_frame("fraction_missing")

plt.figure(figsize=(8, 4))
missing_frac[missing_frac > 0].sort_values().plot(kind="barh", color=sns.color_palette("Set2")[0])
plt.xlabel("Fraction missing (train)")
plt.title("Variables with missing values before imputation")
plt.tight_layout()
plt.show()

# Missingness overview (same library stack as course labs)
msno.matrix(X_train.sample(min(1000, len(X_train)), random_state=42))
plt.title("Missingness matrix (random 1000 train rows)")
plt.show()


### 4.7 Duplicates

Lab exercises emphasize checking duplicate rows after cleaning.


In [ ]:
print("Duplicate rows — train:", X_train.duplicated().sum())
print("Duplicate rows — test:", X_test.duplicated().sum())


### 4.8 Exploratory analysis

We mirror **Lab 5** themes: grouped comparisons and multivariate plots among quantitative variables. A full `pairplot` on tens of thousands of rows is slow; we plot a **random subsample** for visualization only.


In [ ]:
eda = train_df.copy()
eda["high_income"] = encode_target(eda["income"]).map({0: "<=50K", 1: ">50K"})

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

sns.histplot(data=eda, x="age", hue="high_income", stat="density", common_norm=False, ax=axes[0])
axes[0].set_title("Age by income label")

sns.boxplot(data=eda, x="high_income", y="hours-per-week", ax=axes[1])
axes[1].set_title("Hours per week vs income")

sns.boxplot(data=eda, x="high_income", y="education-num", ax=axes[2])
axes[2].set_title("Education years vs income")

plt.tight_layout()
plt.show()


In [ ]:
# Correlation heatmap on numeric columns + binary outcome (train)
num_eda = train_df[NUMERIC].copy()
num_eda["income_gt50"] = y_train.values
corr_matrix = num_eda.corr(numeric_only=True)

plt.figure(figsize=(7, 5))
sns.heatmap(corr_matrix, cmap="vlag", center=0)
plt.title("Correlation matrix — numeric predictors and outcome")
plt.tight_layout()
plt.show()

sample = eda.sample(n=min(1500, len(eda)), random_state=42)
pair_vars = [c for c in NUMERIC if c != "fnlwgt"]
g = sns.pairplot(sample, vars=pair_vars, hue="high_income", corner=True, diag_kind="kde", plot_kws={"alpha": 0.35})
g.figure.suptitle("Pairplot (1500-row subsample; fnlwgt omitted for readability)", y=1.02)
plt.show()


*Write 2–4 sentences here interpreting the EDA figures for your report (what separates >50K vs <=50K in this sample?).*

---

## 5. Results

### 5.1 Pipelines

We build two pipelines that share the same imputation/encoding strategy for fair comparison. Logistic regression scales numeric inputs; the forest uses raw imputed numerics and one-hot dummies.


In [ ]:
numeric_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, NUMERIC),
        ("cat", categorical_pipe, CATEGORICAL),
    ]
)

log_reg = Pipeline(
    steps=[
        ("prep", preprocess),
        ("clf", LogisticRegression(max_iter=2000, random_state=42, solver="lbfgs")),
    ]
)

log_reg.fit(X_train, y_train)


In [ ]:
def evaluate_model(name: str, model, X_te, y_te):
    proba = model.predict_proba(X_te)[:, 1]
    pred = (proba >= 0.5).astype(int)
    acc = accuracy_score(y_te, pred)
    auc = roc_auc_score(y_te, proba)
    print(f"=== {name} ===")
    print(f"Accuracy: {acc:.4f} | ROC-AUC: {auc:.4f}")
    print(classification_report(y_te, pred, target_names=["<=50K", ">50K"]))

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    ConfusionMatrixDisplay.from_predictions(y_te, pred, ax=axes[0], cmap="Blues", colorbar=False)
    axes[0].set_title(f"{name} — confusion matrix")
    RocCurveDisplay.from_predictions(y_te, proba, ax=axes[1])
    axes[1].set_title(f"{name} — ROC curve")
    plt.tight_layout()
    plt.show()


evaluate_model("Logistic regression", log_reg, X_test, y_test)


In [ ]:
rf_preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), NUMERIC),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                ]
            ),
            CATEGORICAL,
        ),
    ]
)

rf_clf = Pipeline(
    steps=[
        ("prep", rf_preprocess),
        (
            "clf",
            RandomForestClassifier(
                n_estimators=200,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

rf_clf.fit(X_train, y_train)
evaluate_model("Random forest", rf_clf, X_test, y_test)


### 5.2 Discussion prompts *(answer in prose)*

- Which model trades off precision/recall differently on the minority `>50K` class?
- Does the nonlinear model justify its extra complexity on this split?

---

## 6. Conclusion

Summarize findings, limitations (1994 snapshot; association vs causation; `fnlwgt` not used as survey weights), and one idea for a follow-up analysis.

---

## 7. References

1. Becker, B., & Kohavi, R. (1996). Adult [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5XW20
2. Pedregosa, F. et al. (2011). Scikit-learn: Machine Learning in Python. *Journal of Machine Learning Research*, 12, 2825–2830.

---
